In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import GBTRegressor
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag, desc, rank, round

active_driver_refs = [
    "max_verstappen", "norris", "piastri", "leclerc", "hamilton", 
    "russell", "alonso", "sainz", "perez", "stroll", "gasly", 
    "albon", "hulkenberg", 'linblad' , "ocon", "bearman", 
    "lawson", "antonelli",'bottas' , "bortoleto", "hadjar"
]


drivers_df = spark.read.table("f1_processed.drivers")
active_drivers = drivers_df.filter(col("driverRef").isin(active_driver_refs))


results_df = spark.read.table("f1_processed.results")
races_df = spark.read.table("f1_processed.races")

base_df = results_df.join(races_df, "race_id") \
    .join(active_drivers, "driver_id") \
    .select("year", "round", "driver_id", "forename", "surname", "points")


driver_window = Window.partitionBy("driver_id").orderBy("year", "round")

ml_data = base_df
for i in range(1, 11):
    ml_data = ml_data.withColumn(f"prev_points_{i}", lag("points", i).over(driver_window))


ml_data = ml_data.dropna()


feature_cols = [f"prev_points_{i}" for i in range(1, 11)]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
ml_ready = assembler.transform(ml_data)

scaler = StandardScaler(inputCol="raw_features", outputCol="features")
scaler_model = scaler.fit(ml_ready)
final_data = scaler_model.transform(ml_ready)


train_df = final_data.filter(col("year") < 2025)
gbt = GBTRegressor(featuresCol="features", labelCol="points", maxIter=25)
model = gbt.fit(train_df)


latest_race_window = Window.partitionBy("driver_id").orderBy(desc("year"), desc("round"))
current_state = final_data.withColumn("rank", rank().over(latest_race_window)) \
    .filter(col("rank") == 1)

predictions = model.transform(current_state)


final_predictions = predictions.select(
    col("forename"), 
    col("surname"), 
    round(col("prediction")).alias("predicted_points_potential")
).orderBy(desc("predicted_points_potential"))

print(" 2025/2026 ACTIVE GRID: PREDICTED PERFORMANCE POTENTIAL")
final_predictions.show()
final_predictions.write.mode('overwrite').saveAsTable('f1_presentation.final_predictions')

 2025/2026 ACTIVE GRID: PREDICTED PERFORMANCE POTENTIAL
+-----------+----------+--------------------------+
|   forename|   surname|predicted_points_potential|
+-----------+----------+--------------------------+
|        Max|Verstappen|                      18.0|
|Andrea Kimi| Antonelli|                      15.0|
|      Lando|    Norris|                      14.0|
|     George|   Russell|                      12.0|
|    Charles|   Leclerc|                       8.0|
|     Carlos|     Sainz|                       7.0|
|      Oscar|   Piastri|                       6.0|
|     Oliver|   Bearman|                       4.0|
|   Fernando|    Alonso|                       3.0|
|       Nico|Hülkenberg|                       3.0|
|       Liam|    Lawson|                       3.0|
|      Isack|    Hadjar|                       3.0|
|      Lewis|  Hamilton|                       2.0|
|    Esteban|      Ocon|                       2.0|
|     Sergio|     Pérez|                       1.0|
|   Valt